# 7.1 (продвинутый уровень)<br>
 Одномерное волновое уравнение: энергия, отражение и стоячие волны. Рассматриваем одномерное волновое уравнение для колебаний струны$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2},\qquad x \in [0, L], \ t \ge 0$$Параметры задачи:<br> длина струны: $L = 1$ <br>скорость распространения волн: $c = 1$
 <br>
Реализовать:<br>1. Выведем простую явную конечно-разностную схему.<br>2. Реализуем **закреплённые концы** $u(0,t) = u(L,t) = 0$.<br>3. Проверим **сохранение полной энергии системы**   $$   E(t) = \int_0^L \left[\left(\frac{\partial u}{\partial t}\right)^2   + c^2 \left(\frac{\partial u}{\partial x}\right)^2 \right] dx   $$4. Реализуем правый **свободный конец**   $$   \frac{\partial u}{\partial x}(L,t) = 0   $$   и проанализируем отражение волны.<br>5. Смоделируем **стоячие волны**, задав начальное условие   $$   u(x,0) = \sin\left(\frac{n\pi x}{L}\right), \qquad   \frac{\partial u}{\partial t}(x,0) = 0   $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Параметрвы задачи
L = 1.0  # длина струны
c = 1.0   # скорость волны

# Численные параметры
Nx = 200   # число отрезков по x -> Nx+1 узлов
dx = L / Nx   # шаг по x
lambda_cfl = 0.9    # c*dt/dx (<= 1)
dt = lambda_cfl * dx / c  # шаг по времени из условия Куранта

Tmax = 2.0      # конечное время моделирования
Nt = int(Tmax / dt)     # количество шагов по времени

x = np.linspace(0, L, Nx + 1)  # сетка по x

print(f"dx = {dx:.4e}, dt = {dt:.4e}, Nt = {Nt}, lambda = {c*dt/dx:.3f}")


In [ ]:
def initial_shape_gaussian(x, x0=0.3, sigma=0.05):
    return np.exp(-((x - x0) ** 2) / (2 * sigma ** 2))

def initial_shape_sine_mode(x, n=1, L=L):
    return np.sin(n * np.pi * x / L)

def initial_velocity_zero(x):
    return np.zeros_like(x)

def prepare_initial_layers(f, g, bc_type='dirichlet'):
    u0 = f(x)
    v0 = g(x)

    # Граничные условия для u0
    if bc_type == 'dirichlet':
        u0[0] = 0.0
        u0[-1] = 0.0
    elif bc_type == 'mixed':
        u0[0] = 0.0  # левый конец закреплён
        u0[-1] = u0[-2]    # правый свободен: du/dx = 0 -> u_N = u_{N-1}

    u1 = np.empty_like(u0)
    lambda2 = (c * dt / dx) ** 2

    # Внутренние точки
    for j in range(1, Nx):
        u1[j] = (
            u0[j]
            + dt * v0[j]
            + 0.5 * lambda2 * (u0[j + 1] - 2 * u0[j] + u0[j - 1])
        )

    # Границы для u1
    if bc_type == 'dirichlet':
        u1[0] = 0.0
        u1[-1] = 0.0
    elif bc_type == 'mixed':
        u1[0] = 0.0
        u1[-1] = u1[-2]

    return u0, u1


In [ ]:
def step_dirichlet(u_prev, u_curr):
    u_next = np.empty_like(u_curr)
    lambda2 = (c * dt / dx) ** 2

    # внутренние узлы
    for j in range(1, Nx):
        u_next[j] = (
            2 * u_curr[j]
            - u_prev[j]
            + lambda2 * (u_curr[j + 1] - 2 * u_curr[j] + u_curr[j - 1])
        )

    # границы
    u_next[0] = 0.0
    u_next[-1] = 0.0

    return u_next


def step_mixed(u_prev, u_curr):
    u_next = np.empty_like(u_curr)
    lambda2 = (c * dt / dx) ** 2

    #внутренние узлы до предпоследнего
    for j in range(1, Nx):
        u_next[j] = (
            2 * u_curr[j]
            - u_prev[j]
            + lambda2 * (u_curr[j + 1] - 2 * u_curr[j] + u_curr[j - 1])
        )

    # левый конец
    u_next[0] = 0.0

    # правый конец: du/dx = 0 -> u_N = u_{N-1}
    u_next[-1] = u_next[-2]

    return u_next


## Дискретная энергия системы<br>
Непрерывная энергия:$$E(t) = \int_0^L \left[\left(\frac{\partial u}{\partial t}\right)^2+ c^2 \left(\frac{\partial u}{\partial x}\right)^2\right] dx$$<br>
Аппроксимации:производная по времени:  $$  \frac{\partial u}{\partial t}(x_j, t^n)  \approx \frac{u_j^n - u_j^{n-1}}{\Delta t}  $$<br>
 производная по пространству:  $$  \frac{\partial u}{\partial x}(x_j, t^n)  \approx \frac{u_{j+1}^n - u_{j-1}^n}{2\Delta x}  $$<br>
 Тогда дискретная энергия:$$E^n \approx\sum_{j=1}^{N-1} \left[\left(\frac{u_j^n - u_j^{n-1}}{\Delta t}\right)^2+ c^2 \left(\frac{u_{j+1}^n - u_{j-1}^n}{2\Delta x}\right)^2\right] \Delta x$$

In [ ]:
def compute_energy(u_prev, u_curr, bc_type='dirichlet'):
    # du/dt
    du_dt = (u_curr - u_prev) / dt

    # du/dx
    du_dx = np.zeros_like(u_curr)
    du_dx[1:-1] = (u_curr[2:] - u_curr[:-2]) / (2 * dx)

    if bc_type == 'mixed':
        # на свободном конце берём одностороннюю разность
        du_dx[-1] = (u_curr[-1] - u_curr[-2]) / dx

    energy_density = du_dt**2 + (c * du_dx) ** 2

    # интеграл по x -> сумма * dx (берём ток внутренние узлы)
    E = np.sum(energy_density[1:-1]) * dx
    return E


## Закрепленныве концы

In [ ]:
# Начальные условия
f = lambda x: initial_shape_gaussian(x, x0=0.3, sigma=0.05)
g = initial_velocity_zero

u_prev, u_curr = prepare_initial_layers(f, g, bc_type='dirichlet')

energies = [compute_energy(u_prev, u_curr, bc_type='dirichlet')]
times = [0.0]

# Моменты времени, в которые хотим видеть профиль
snapshots_t = [0.0, 0.5, 1.0, 1.5, 2.0]
snapshots_u = {t: None for t in snapshots_t}
snapshots_u[0.0] = u_prev.copy()

for n in range(1, Nt):
    t = n * dt
    u_next = step_dirichlet(u_prev, u_curr)

    E = compute_energy(u_curr, u_next, bc_type='dirichlet')
    energies.append(E)
    times.append(t)

    for ts in snapshots_t:
        if snapshots_u[ts] is None and abs(t - ts) < dt / 2:
            snapshots_u[ts] = u_curr.copy()

    u_prev, u_curr = u_curr, u_next

# если не попали точно во время — возьмём последнее значение
for ts in snapshots_t:
    if snapshots_u[ts] is None:
        snapshots_u[ts] = u_curr.copy()



In [ ]:
# Энергия во времени (закреплённые концы)
plt.figure(figsize=(6, 4))
plt.plot(times, energies)
plt.xlabel("t")
plt.ylabel("E(t)")
plt.title("Энергия системы (u(0,t)=u(L,t)=0)")
plt.grid(True)
plt.show()

print(f"Мин энергия: {np.min(energies):.6f}")
print(f"Макс энергия: {np.max(energies):.6f}")
print(f"Относительное изменение: {(max(energies)-min(energies))/energies[0]:.2e}")


In [ ]:
# профили u(x,t) в разные моменты времени
plt.figure(figsize=(7, 4))
for ts in snapshots_t:
    plt.plot(x, snapshots_u[ts], label=f"t = {ts:.2f}")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("Отражение волны при закреплённых концах")
plt.grid(True)
plt.legend()
plt.show()


## Свободный правый конец

In [ ]:
# те же нач условия, но правый конец свободен
f = lambda x: initial_shape_gaussian(x, x0=0.3, sigma=0.05)
g = initial_velocity_zero

u_prev, u_curr = prepare_initial_layers(f, g, bc_type='mixed')

energies_mixed = [compute_energy(u_prev, u_curr, bc_type='mixed')]
times_mixed = [0.0]

snapshots_u_mixed = {t: None for t in snapshots_t}
snapshots_u_mixed[0.0] = u_prev.copy()

for n in range(1, Nt):
    t = n * dt
    u_next = step_mixed(u_prev, u_curr)

    E = compute_energy(u_curr, u_next, bc_type='mixed')
    energies_mixed.append(E)
    times_mixed.append(t)

    for ts in snapshots_t:
        if snapshots_u_mixed[ts] is None and abs(t - ts) < dt / 2:
            snapshots_u_mixed[ts] = u_curr.copy()

    u_prev, u_curr = u_curr, u_next

for ts in snapshots_t:
    if snapshots_u_mixed[ts] is None:
        snapshots_u_mixed[ts] = u_curr.copy()



In [ ]:
# Ээt.figure(figsize=(6, 4))
plt.plot(times_mixed, energies_mixed)
plt.xlabel("t")
plt.ylabel("E(t)")
plt.title("Энергия (левый закреплён, правый свободен)")
plt.grid(True)
plt.show()

print(f"Мин энергия: {np.min(energies_mixed):.6f}")
print(f"Макс энергия: {np.max(energies_mixed):.6f}")
print(f"Относительное изменение: {(max(energies_mixed)-min(energies_mixed))/energies_mixed[0]:.2e}")


In [ ]:
# сравнение профилей для закреплённого и свободного конца
plt.figure(figsize=(7, 4))
for ts in snapshots_t:
    plt.plot(x, snapshots_u[ts], '--', label=f"fixed, t={ts:.2f}")
    plt.plot(x, snapshots_u_mixed[ts], label=f"free,  t={ts:.2f}")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("Сравнение отражения: закреплённый vs свободный правый конец")
plt.grid(True)
plt.legend(loc='upper right', fontsize=7)
plt.show()


## Стоячие волны
    
Для струны с закреплёнными концами $u(0,t)=u(L,t)=0$ собственные моды имеют вид$$u_n(x,t) = \sin\left(\frac{n\pi x}{L}\right)\cos\left(\frac{n\pi c t}{L}\right),\qquad n = 1,2,3..$$Если задать начальные условия$$u(x,0) = \sin\left(\frac{n\pi x}{L}\right),\qquad\frac{\partial u}{\partial t}(x,0) = 0,$$то получим **чистую стоячую волну** с $n-1$ внутренними узлами.

In [ ]:
# Номер моды
n_mode = 2

f = lambda x: initial_shape_sine_mode(x, n=n_mode, L=L)
g = initial_velocity_zero

u_prev, u_curr = prepare_initial_layers(f, g, bc_type='dirichlet')

snapshots_t_modes = [0.0, 0.25, 0.5, 0.75, 1.0]
snapshots_modes = {t: None for t in snapshots_t_modes}
snapshots_modes[0.0] = u_prev.copy()

for n in range(1, Nt):
    t = n * dt
    u_next = step_dirichlet(u_prev, u_curr)

    for ts in snapshots_t_modes:
        if snapshots_modes[ts] is None and abs(t - ts) < dt / 2:
            snapshots_modes[ts] = u_curr.copy()

    u_prev, u_curr = u_curr, u_next

for ts in snapshots_t_modes:
    if snapshots_modes[ts] is None:
        snapshots_modes[ts] = u_curr.copy()


In [ ]:
plt.figure(figsize=(7, 4))
for ts in snapshots_t_modes:
    plt.plot(x, snapshots_modes[ts], label=f"t = {ts:.2f}")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title(f"Стоячая волна, мода n = {n_mode}")
plt.grid(True)
plt.legend()
plt.show()
